## 1) Imports and helpers

In [1]:
# Core imports
import os
import itertools
from datetime import datetime
import pandas as pd
import jax
import jax.numpy as jnp

# Import the framework helpers used by the CLI runner
from framework.registry import ComparisonRegistry
from framework.runner import run_adapter_benchmark
from framework.adapters import setup_bbob_instances

# toml loader (py3.11 has tomllib; otherwise tomli)
try:
    import tomllib
except Exception:
    import tomli as tomllib  # type: ignore
    
# write export CUDA_VISIBLE_DEVICES=0 in the terminal before running this script
os.environ['CUDA_VISIBLE_DEVICES'] = '0'


print(f'JAX: {jax.__version__} | Backend: {jax.default_backend()} | Device: {jax.devices()[0].device_kind}')

JAX: 0.8.0 | Backend: cpu | Device: cpu


## 1b) HLO Extraction Helper
Import the HLO extraction function for compiler analysis.

In [2]:
from framework.runner import extract_hlo_from_adapter

## 2) Experiment configuration (interactive)
Adjust values here instead of passing a CLI config file.

In [3]:
# Experiment metadata
EXP_NAME = 'notebook_benchmark'
OUTPUT_DIR = 'results/notebook_runs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Grid-like configuration (similar to CLI's toml grid)
# Print available algorithms for user reference
print('Available algorithms:', list(ComparisonRegistry._registry.keys()))
grid = {
    'algorithms': ['Standard_GA', 'Standard_GA_Ablation'],#, 'MR15_GA', 'Differential_Evolution'],  # Now includes DE!
    'tasks': ['rastrigin'],
    'dimensions': [20],
    'pop_sizes': [100],
    'unroll_factors': [1, 25],
    'generations': 100,
    'repeats': 1,  # lower default for quick notebook runs
    'seeds': [42],
}

# Optional hyperparams that will be merged with algorithm defaults
hyperparams = {}

print('Configured experiment grid:')
for k, v in grid.items():
    print(f'  {k}: {v}')

Available algorithms: ['Standard_GA', 'Standard_GA_Ablation', 'MR15_GA', 'Differential_Evolution']
Configured experiment grid:
  algorithms: ['Standard_GA', 'Standard_GA_Ablation']
  tasks: ['rastrigin']
  dimensions: [20]
  pop_sizes: [100]
  unroll_factors: [1, 25]
  generations: 100
  repeats: 1
  seeds: [42]


## 3) Build job queue
This creates the tuple list the CLI would iterate over. You can subset or preview it before running.

In [4]:
# Build the job queue (cartesian product)
job_queue = list(itertools.product(
    grid['algorithms'], grid['tasks'], grid['dimensions'], grid['pop_sizes'], grid.get('unroll_factors', [1])
))
print(f'Total configurations: {len(job_queue)}')
# Preview first few
job_queue[:5]

Total configurations: 4


[('Standard_GA', 'rastrigin', 20, 100, 1),
 ('Standard_GA', 'rastrigin', 20, 100, 25),
 ('Standard_GA_Ablation', 'rastrigin', 20, 100, 1),
 ('Standard_GA_Ablation', 'rastrigin', 20, 100, 25)]

## 4) Single-job runner function
Encapsulates the logic from `benchmarks/cli.py` for one (algo,task,dim,pop,unroll) tuple.

In [ ]:
def run_job(algo_key, task, dim, pop, unroll, master_seed, generations, repeats, hypers):
    """Run adapters for the given job and return packaged result dicts.
    Returns a list with result dicts (one or two depending on whether Evosax is available).
    """
    spec = ComparisonRegistry.get(algo_key)
    # Merge default hypers with provided ones
    merged_hypers = {**spec.default_hypers, **(hypers or {})}

    # Setup problem instances (MalthusJAX / Evosax adapters expect different objects)
    m_eval, e_prob = setup_bbob_instances(task, dim, master_seed)

    # Factories from registry create adapters given hyperparams etc.
    m_adapter = spec.malthus_factory(pop, dim, master_seed, merged_hypers, m_eval)

    # Run MalthusJAX benchmark
    res_m = run_adapter_benchmark(m_adapter, generations, master_seed, 'MalthusJAX', pop, unroll, repeats)

    base = {
        'Algorithm': algo_key, 'Task': task, 'Dim': dim, 'Pop_Size': pop,
        'Unroll': unroll, 'Gens': generations
    }

    def package(res):
        # Use absolute value for normalized fitness comparison
        # MalthusJAX uses maximization (positive), Evosax uses minimization (negative)
        raw_fit = getattr(res, 'best_fitness_final', None)
        normalized_fitness = abs(raw_fit) if raw_fit is not None else None
        return {
            **base,
            'Framework': res.framework,
            'Mean_GPS': getattr(res, 'mean_gps', None),
            'Mean_Time': getattr(res, 'mean_exec_time', None),
            "Std_Time": getattr(res, 'std_exec_time', None),
            'Compile_Time': getattr(res, 'compile_time', None),
            'Best_Fitness': raw_fit,
            'Fitness_Std': getattr(res, 'fitness_std', None),
            'Normalized_Fitness': normalized_fitness,
        }

    results = [package(res_m)]
    
    # Only run Evosax if the spec includes it
    if spec.evosax_factory is not None:
        e_adapter = spec.evosax_factory(pop, dim, master_seed, merged_hypers, e_prob)
        res_e = run_adapter_benchmark(e_adapter, generations, master_seed, 'Evosax', pop, unroll, repeats)
        results.append(package(res_e))
        
        # Print speedup and fitness comparison using absolute values
        speedup = res_m.mean_gps / res_e.mean_gps
        mjx_fit_norm = abs(res_m.best_fitness_final)
        evosax_fit_norm = abs(res_e.best_fitness_final)
        fit_diff = abs(mjx_fit_norm - evosax_fit_norm)
        fit_match = "✓" if fit_diff < 1.0 else "✗"
        print(f"   >>> Speedup: {speedup:.2f}x (MJX: {res_m.mean_gps:.2f} GPS | Evosax: {res_e.mean_gps:.2f} GPS)")
        print(f"   >>> Fitness (|val|): MJX={mjx_fit_norm:.2e}±{res_m.fitness_std:.2e} | Evosax={evosax_fit_norm:.2e}±{res_e.fitness_std:.2e} {fit_match}")
    else:
        print(f"   >>> [Malthus-Only] Mean GPS: {res_m.mean_gps:.2f}")
        print(f"   >>> Fitness: {abs(res_m.best_fitness_final):.2e}±{res_m.fitness_std:.2e}")

    return results

## 5) Execute the queue (interactive run)
Control the number of jobs to run so the notebook stays responsive. Start with a subset for learning.

In [6]:
# Quick controls: run_all=True will run the entire job_queue (careful).
run_all = True
max_jobs = 2  # when run_all=False, only run first `max_jobs` entries

master_seed = grid['seeds'][0] if grid.get('seeds') else 0
generations = grid['generations']
repeats = grid.get('repeats', 30)

results = []
jobs_to_run = job_queue if run_all else job_queue[:max_jobs]

for i, (algo, task, dim, pop, unroll) in enumerate(jobs_to_run, 1):
    print(f'Running job {i}/{len(jobs_to_run)}: Algo={algo}, Task={task}, Dim={dim}, Pop={pop}, Unroll={unroll}')
    try:
        packaged = run_job(algo, task, dim, pop, unroll, master_seed, generations, repeats, hyperparams)
        results.extend(packaged)
    except Exception as e:
        print('ERROR running job:', e)

# Convert to DataFrame
df = pd.DataFrame(results)
df

Running job 1/4: Algo=Standard_GA, Task=rastrigin, Dim=20, Pop=100, Unroll=1
[MalthusJAX] Compiling (Unroll=1)... Done (0.3353s)
[Evosax] Compiling (Unroll=1)... Done (0.2854s)
   >>> Speedup: 1.18x (MJX: 7949.36 GPS | Evosax: 6719.77 GPS)
Running job 2/4: Algo=Standard_GA, Task=rastrigin, Dim=20, Pop=100, Unroll=25
[MalthusJAX] Compiling (Unroll=25)... Done (3.2858s)
[Evosax] Compiling (Unroll=25)... Done (3.1699s)
   >>> Speedup: 1.04x (MJX: 6522.50 GPS | Evosax: 6281.87 GPS)
Running job 3/4: Algo=Standard_GA_Ablation, Task=rastrigin, Dim=20, Pop=100, Unroll=1
[MalthusJAX] Compiling (Unroll=1)... Done (0.4025s)
   >>> [Malthus-Only] Mean GPS: 7484.26
Running job 4/4: Algo=Standard_GA_Ablation, Task=rastrigin, Dim=20, Pop=100, Unroll=25
[MalthusJAX] Compiling (Unroll=25)... Done (5.1268s)
   >>> [Malthus-Only] Mean GPS: 7031.63


,Algorithm,Task,Dim,Pop_Size,Unroll,Gens,Framework,Mean_GPS,Mean_Time,Std_Time,Compile_Time,Best_Fitness
0,Standard_GA,rastrigin,20,100,1,100,MalthusJAX,7949.362556,0.012580,0.0,0.335252,72.395294
1,Standard_GA,rastrigin,20,100,1,100,Evosax,6719.771671,0.014881,0.0,0.285395,-30.124359
2,Standard_GA,rastrigin,20,100,25,100,MalthusJAX,6522.501573,0.015332,0.0,3.285836,72.395294
3,Standard_GA,rastrigin,20,100,25,100,Evosax,6281.867487,0.015919,0.0,3.169942,-30.124359
4,Standard_GA_Ablation,rastrigin,20,100,1,100,MalthusJAX,7484.259687,0.013361,0.0,0.402529,114.200752
5,Standard_GA_Ablation,rastrigin,20,100,25,100,MalthusJAX,7031.627323,0.014221,0.0,5.126845,114.200752


## 6) Save & inspect results
Save a CSV copy and display summary statistics.

In [7]:
# Save results with timestamp
if not df.empty:
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    out_file = os.path.join(OUTPUT_DIR, f'notebook_benchmark_{ts}.csv')
    df.to_csv(out_file, index=False)
    print('Saved:', out_file)
    display(df.describe(include='all'))
else:
    print('No results to save (df is empty).')

Saved: results/notebook_runs/notebook_benchmark_20260106_182310.csv


,Algorithm,Task,Dim,Pop_Size,Unroll,Gens,Framework,Mean_GPS,Mean_Time,Std_Time,Compile_Time,Best_Fitness
count,6,6,6.0,6.0,6.000000,6.0,6,6.000000,6.000000,6.0,6.000000,6.000000
unique,2,1,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN
top,Standard_GA,rastrigin,NaN,NaN,NaN,NaN,MalthusJAX,NaN,NaN,NaN,NaN,NaN
freq,4,6,NaN,NaN,NaN,NaN,4,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,20.0,100.0,13.000000,100.0,NaN,6998.231716,0.014382,0.0,2.100967,52.157229
std,NaN,NaN,0.0,0.0,13.145341,0.0,NaN,625.943321,0.001251,0.0,2.049453,66.420593
min,NaN,NaN,20.0,100.0,1.000000,100.0,NaN,6281.867487,0.012580,0.0,0.285395,-30.124359
25%,NaN,NaN,20.0,100.0,1.000000,100.0,NaN,6571.819098,0.013576,0.0,0.352072,-4.494446
50%,NaN,NaN,20.0,100.0,13.000000,100.0,NaN,6875.699497,0.014551,0.0,1.786236,72.395294
75%,NaN,NaN,20.0,100.0,25.000000,100.0,NaN,7371.101596,0.015219,0.0,3.256862,103.749388


## Notes and next steps
- Use `run_all = True` to run the full grid (may be long).
- Adjust `grid` or `hyperparams` cells to explore different setups.
- The notebook mirrors `benchmarks/cli.py` but keeps execution interactive and educational.

## 7) Extract HLO (Compiler IR) - Optional
Use this to analyze the compiled representation without running benchmarks.

In [8]:
# Example: Extract HLO for a specific configuration
# Try 'Standard_GA', 'Standard_GA_Ablation', 'MR15_GA', or 'Differential_Evolution'
algo_key = 'Standard_GA_Ablation'
task = 'rastrigin'
framework = 'MalthusJAX'  # 'Evosax' or 'MalthusJAX'
dim = 20
pop = 100
unroll = 25
seed = 42

# Setup the adapter
spec = ComparisonRegistry.get(algo_key)
merged_hypers = {**spec.default_hypers, **hyperparams}
m_eval, e_prob = setup_bbob_instances(task, dim, seed)

# Create MalthusJAX adapter
m_adapter = spec.malthus_factory(pop, dim, seed, merged_hypers, m_eval)

# Create Evosax adapter only if available
e_adapter = None
if spec.evosax_factory is not None:
    e_adapter = spec.evosax_factory(pop, dim, seed, merged_hypers, e_prob)

# Select adapter based on framework choice
if framework == 'MalthusJAX':
    adapter = m_adapter
elif framework == 'Evosax' and e_adapter is not None:
    adapter = e_adapter
else:
    raise ValueError(f"Framework '{framework}' not available for algorithm '{algo_key}'")

# Extract HLO and save to file
hlo_output_path = f'{OUTPUT_DIR}/new_hlo_{algo_key}_{task}_d{dim}_p{pop}_u{unroll}_{framework}.txt'
hlo_text = extract_hlo_from_adapter(
    adapter,
    num_gens=100, 
    seed=seed,
    framework_name=framework,
    unroll_factor=unroll,
    output_path=hlo_output_path
)

print(f"\nHLO text length: {len(hlo_text)} characters")
print(f"First 500 chars:\n{hlo_text[:500]}")


[MalthusJAX] Compiling (Unroll=25) to extract HLO... Done (4.6802s)
HLO saved to: results/notebook_runs/new_hlo_Standard_GA_Ablation_rastrigin_d20_p100_u25_MalthusJAX.txt

HLO text length: 21810870 characters
First 500 chars:
HloModule jit_scan_loop, is_scheduled=true, entry_computation_layout={(f32[100,20]{1,0}, f32[100]{0}, f32[20]{0}, s32[], f32[], /*index=5*/s32[], u32[2]{0})->(f32[100,20]{1,0}, f32[100]{0}, f32[20]{0}, s32[], f32[], /*index=5*/s32[], u32[2]{0}, f32[100]{0}, f32[100]{0}, s32[100]{0}, /*index=10*/u32[100,2]{1,0})}, allow_spmd_sharding_propagation_to_parameters={false,false,false,true,false,true,true}, allow_spmd_sharding_propagation_to_output={true,true,true,true,true,true,true,true,true,true,true
